In [19]:
from typing import Any
import os
import pandas as pd
from ipywidgets import HTML, VBox
from cosmograph import cosmo
from qnet.actant.future import load_corpus, Corpus
from qnet.config import c_env
import networkx as nx
import math
import numpy as np

In [2]:
corpus = load_corpus(c_env.ACTANT_FUTURE)


In [3]:
import graphistry
graphistry.register(api=3, username=os.getenv('GRAPHISTRY_USERNAME'), password=os.getenv('GRAPHISTRY_PASSWORD'), protocol='https', server='hub.graphistry.com')

In [ ]:
def corpus_to_bipartite(corpus) -> tuple[pd.DataFrame, pd.DataFrame]:
    # String IDs avoid type drift issues from the widget
    actants: list[str] = list(corpus.Actants.keys())
    actant_pid: dict[str, str] = {a: f"A:{a}" for a in actants}

    points_rows: list[dict] = []
    links_rows: list[dict] = []

    # Actant nodes
    for a in actants:
        points_rows.append({
            "id": actant_pid[a],
            "node_type": "Actant",
            "label": a,
            "role": corpus.Actants[a],  # Actor | Constructor | Constraint
        })

    # Utterance nodes + incidence links
    for i, u in enumerate(corpus.Utterances, start=1):
        uid = f"U:{i}"
        refs = [".".join(r) for r in u.refs]
        snippet = (u.text[:80] + "…") if len(u.text) > 80 else u.text
        points_rows.append({
            "id": uid,
            "node_type": "Utterance",
            "label": snippet,
            "snippet": snippet,
            "full_text": u.text,
            "refs": refs,
            "utterance_idx": i,
        })
        for a in u.actants:
            if a in actant_pid:
                links_rows.append({"source": uid, "target": actant_pid[a], "value": 1.0})

    points = pd.DataFrame(points_rows)
    links = pd.DataFrame(links_rows)
    return points, links

points, links = corpus_to_bipartite(corpus)

# Uniform sizes; keep colors by type and labels by label
widget = cosmo(
    points=points,
    links=links,
    point_id_by="id",
    link_source_by="source",
    link_target_by="target",
    point_color_by="node_type",
    point_label_by="label",
    point_cluster_by="node_type",
    show_hovered_point_label=True,
    point_include_columns=["node_type","role","snippet","full_text","refs","utterance_idx"],
    link_include_columns=["value"],
    # uniform sizing:
    point_size=6,                         # constant size; see docs
    fit_view_on_init=True,
    disable_point_size_legend=True,
)

# --- Robust click details (string IDs; guard if not present) ---
details = HTML("<i>click a node…</i>")
pidx = points.set_index("id", drop=False)  # O(1) id lookup

def on_click(change):
    pid = change["new"]
    if not pid or pid not in pidx.index:
        return
    row = pidx.loc[pid]
    if row["node_type"] == "Utterance":
        act_ids = links.loc[links["source"] == pid, "target"].unique().tolist()
        act_names = [pidx.loc[a, "label"] for a in act_ids if a in pidx.index]
        details.value = f"""
        <b>Utterance {int(row['utterance_idx'])}</b><br/>
        <b>Refs:</b> {', '.join(row['refs']) if isinstance(row['refs'], list) else row['refs']}<br/>
        <b>Actants:</b> {', '.join(act_names)}<br/>
        <pre style="white-space:pre-wrap;margin-top:6px;">{row['full_text']}</pre>
        """
    else:
        utt_ids = links.loc[links["target"] == pid, "source"].unique().tolist()
        snippets = [pidx.loc[u, "label"] for u in utt_ids if u in pidx.index]
        role = row.get("role", "")
        details.value = f"""
        <b>Actant:</b> {row['label']} <b>({role})</b><br/>
        <b>Utterances:</b> {', '.join(snippets[:12])}{' …' if len(snippets) > 12 else ''}
        """

widget.observe(on_click, names=["clicked_point_id"])
VBox([widget, details])


In [14]:
import colorsys

def _brighten(h: str, t: float) -> str:
    h = h.lstrip('#')
    r, g, b = int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16)
    r, g, b = [x / 255.0 for x in (r, g, b)]
    h_, l, s = colorsys.rgb_to_hls(r, g, b)
    l = min(1.0, l * (1 + t))
    s = min(1.0, s * (1 + t))
    r, g, b = colorsys.hls_to_rgb(h_, l, s)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

def to_node_edges(c: Corpus, anon: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    def _interp(x: str, y: str, t: float) -> str:
        x, y = x.lstrip('#'), y.lstrip('#')
        ar, ag, ab = int(x[:2],16), int(x[2:4],16), int(x[4:],16)
        br, bg, bb = int(y[:2],16), int(y[2:4],16), int(y[4:],16)
        r = int(ar + (br - ar) * t)
        g = int(ag + (bg - ag) * t)
        b = int(ab + (bb - ab) * t)
        return f'#{r:02x}{g:02x}{b:02x}'

    # --- brightened palettes
    BRIGHT = 0.2
    ROLE_HEX = {
        "Actor":       _brighten("#4b729b", BRIGHT),
        "Constructor": _brighten("#5a8545", BRIGHT),
        "Constraint":  _brighten("#408583", BRIGHT),
    }
    LOW  = _brighten("#5b1c3a", BRIGHT)
    HIGH = _brighten("#b370bd", BRIGHT)

    # discrete affect bins → color map
    color_map: dict[str, str] = {
        f"Actant:{role}": col for role, col in ROLE_HEX.items()
    }
    for idx, k in enumerate(range(-4, 5)):  # -4..+4
        t = idx / 8
        color_map[f"Utterance:{k}"] = _interp(LOW, HIGH, t)

    # --- ids & labels
    actant_label: dict[str, str] = {a: (f"Actant {i:03d}" if anon else a)
                                    for i, a in enumerate(c.Actants.keys(), 1)}
    actant_id: dict[str, str] = {a: f"A::{actant_label[a]}" for a in c.Actants.keys()}

    nodes: list[dict[str, Any]] = []
    edges: list[dict[str, Any]] = []

    # actant nodes
    for a, role in c.Actants.items():
        ck = f"Actant:{role}"
        nodes.append({
            "node_id": actant_id[a],
            "type": "Actant",
            "role": role,
            "label": actant_label[a],
            "size": 30,
            "color_key": ck,
            "color_hex": color_map[ck],
        })

    # utterances + edges
    for i, u in enumerate(c.Utterances, 1):
        uid = f"U::{i}"
        snippet = f"Utterance {i:03d}" if anon else (u.text[:96] + "…") if len(u.text) > 96 else u.text
        full_text = f"Dummy text for utterance {i:03d}" if anon else u.text
        k = max(-4, min(4, int(u.affect)))
        ck = f"Utterance:{k}"

        nodes.append({
            "node_id": uid,
            "type": "Utterance",
            "role": "Utterance",
            "label": snippet,
            "full_text": full_text,
            "story": u.story,
            "variant": u.variant,
            "pa": int(u.pa),
            "na": int(u.na),
            "affect": k,
            "utterance_idx": i,
            "size": 30,
            "color_key": ck,
            "color_hex": color_map[ck],
        })

        for a in u.actants:
            if a in actant_id:
                edges.append({"src": uid, "dst": actant_id[a], "etype": "MENTIONS"})

    nodes_df = pd.DataFrame(nodes)
    edges_df = pd.DataFrame(edges)

    # --- easy metrics (directed)
    G = nx.DiGraph()
    G.add_nodes_from(nodes_df["node_id"])
    G.add_edges_from(edges_df[["src", "dst"]].itertuples(index=False, name=None))

    pr = nx.pagerank(G, alpha=0.85) if G.number_of_edges() else {n: 0.0 for n in G.nodes()}
    try:
        hubs, auths = nx.hits(G, max_iter=200, tol=1e-8)
    except Exception:
        hubs = {n: 0.0 for n in G.nodes()}
        auths = {n: 0.0 for n in G.nodes()}

    nodes_df["pr"] = nodes_df["node_id"].map(pr).astype(float)
    nodes_df["hub"] = nodes_df["node_id"].map(hubs).astype(float)
    nodes_df["auth"] = nodes_df["node_id"].map(auths).astype(float)

    return nodes_df, edges_df


In [15]:
nodes_df, edges_df = to_node_edges(corpus)  # or anon=False

color_map = {r["color_key"]: r["color_hex"]
             for _, r in nodes_df[["color_key", "color_hex"]].drop_duplicates().iterrows()}

g = (
    graphistry
      .bind(source='src', destination='dst', node='node_id', point_title='label')
      .nodes(nodes_df)
      .edges(edges_df)
      .encode_point_size('pr')
      .encode_point_color('color_key', categorical_mapping=color_map) # default_mapping='#7e7d75'
      .name('Utterances × Actants')
      .description('Bipartite: utterance ↔ actants; colors by role & affect')
      .settings(url_params={'pointsOfInterestMax': 16})
)

g.plot(render='url')

'https://hub.graphistry.com/graph/graph.html?dataset=73f31c4c28034df2a71b4765e2986183&type=arrow&viztoken=6c3f1d01-fbfa-460a-8bac-3ff25ac2ba23&usertag=7ab2484f-pygraphistry-0.41.2&splashAfter=1756699569&info=true&pointsOfInterestMax=16'

# Valence Lift

In [22]:
ALPHA: float = 0.85

def valence_lift(c: Corpus, alpha: float = ALPHA) -> tuple[pd.DataFrame, pd.DataFrame]:
    a_id: dict[str, str] = {a: f"A::{a}" for a in c.Actants}
    u_ids: list[str] = [f"U::{i}" for i in range(1, len(c.Utterances) + 1)]
    all_ids: list[str] = u_ids + list(a_id.values())

    # edges: U -> A (display); PPR uses both directions
    edges: list[dict[str, Any]] = []
    for i, u in enumerate(c.Utterances, 1):
        uid = f"U::{i}"
        for a in u.actants:
            if a in a_id:
                edges.append({"src": uid, "dst": a_id[a], "etype": "MENTIONS"})

    # PPR graph
    G = nx.DiGraph()
    G.add_nodes_from(all_ids)
    for e in edges:
        G.add_edge(e["src"], e["dst"])
        G.add_edge(e["dst"], e["src"])

    # seeds
    def _norm(d: dict[str, float]) -> dict[str, float]:
        s = sum(d.values()) or 1.0
        return {k: v / s for k, v in d.items()}

    pers_pos = _norm({f"U::{i}": float(u.pa) for i, u in enumerate(c.Utterances, 1)})
    pers_neg = _norm({f"U::{i}": float(u.na) for i, u in enumerate(c.Utterances, 1)})
    base_u   = _norm({uid: 1.0 for uid in u_ids})
    base_all = _norm({nid: 1.0 for nid in all_ids})

    pr_pos = nx.pagerank(G, alpha=alpha, personalization=pers_pos, dangling=base_u)
    pr_neg = nx.pagerank(G, alpha=alpha, personalization=pers_neg, dangling=base_u)
    pr_bu  = nx.pagerank(G, alpha=alpha, personalization=base_u,   dangling=base_u)
    pr_ba  = nx.pagerank(G, alpha=alpha, personalization=base_all, dangling=base_all)

    # node rows
    rows: list[dict[str, Any]] = []
    for nid in all_ids:
        p = float(pr_pos.get(nid, 0.0))
        n = float(pr_neg.get(nid, 0.0))
        bu = float(pr_bu.get(nid, 0.0))
        ba = float(pr_ba.get(nid, 0.0))

        p_lift_u = p / bu if bu > 0 else 0.0
        n_lift_u = n / bu if bu > 0 else 0.0
        p_lift   = p / ba if ba > 0 else 0.0
        n_lift   = n / ba if ba > 0 else 0.0

        val             = p - n
        valence_lift    = p_lift - n_lift
        mag_lift        = math.sqrt(p_lift * p_lift + n_lift * n_lift)
        balance         = (p_lift - n_lift) / (p_lift + n_lift) if (p_lift + n_lift) > 0 else 0.0
        tension_lift    = 2.0 * min(p_lift, n_lift)

        rows.append({
            "node_id": nid,
            "p_rank": p, "n_rank": n, "valence": val,
            "p_lift": p_lift, "n_lift": n_lift,       # all-nodes baseline (works for both types)
            "p_lift_u": p_lift_u, "n_lift_u": n_lift_u,  # U-only baseline (legacy feel)
            "valence_lift": valence_lift,
            "mag_lift": mag_lift,
            "balance": balance,           # [-1, 1]
            "tension_lift": tension_lift  # co-activation
        })

    ndf = pd.DataFrame(rows).set_index("node_id", drop=False)

    # enrich with labels & metadata
    for i, u in enumerate(c.Utterances, 1):
        uid = f"U::{i}"
        ndf.loc[uid, ["type", "role", "label", "story", "variant", "pa", "na", "affect", "utterance_idx"]] = [
            "Utterance", "Utterance",
            (u.text[:96] + "…") if len(u.text) > 96 else u.text,
            u.story, u.variant, int(u.pa), int(u.na), int(u.affect), i
        ]

    for a, role in c.Actants.items():
        aid = a_id[a]
        if aid in ndf.index:
            ndf.loc[aid, ["type", "role", "label"]] = ["Actant", role, a]

    # defaults for viz
    ndf["color_metric"] = ndf["valence_lift"]  # alt: 'balance' or 'mag_lift'
    ndf["size"] = np.where(ndf["type"] == "Utterance", 24.0, 30.0)

    return ndf.reset_index(drop=True), pd.DataFrame(edges)


In [24]:
# PALETTE: list[str] = ['#1A666C', '#2D7EA3', '#4C8BD5', '#7592FC', '#A498FF', '#D09FFF', '#F3ADFF', '#FFC1FF']
PALETTE = ['#8E3E29', '#A15D28', '#A87F2C', '#A7A13C', '#A5C158', '#A8DB80']

nodes_df, edges_df = valence_lift(corpus)

metric: str = "valence_lift"   # try: 'balance', 'mag_lift'
size_metric: str = "mag_lift"

g = (
    graphistry
      .bind(source='src', destination='dst', node='node_id', point_title='label')
      .nodes(nodes_df)
      .edges(edges_df)
      .encode_point_size(size_metric)
      .encode_point_color(metric, palette=PALETTE, as_continuous=True)
      .name('Utterances × Actants — valence lift')
      .settings(url_params={'pointsOfInterestMax': 16})
)

g.plot(render='url')


'https://hub.graphistry.com/graph/graph.html?dataset=0af98c3cb3894080ae39a2e1bde11ca6&type=arrow&viztoken=7441803a-85e7-4eb7-b7cc-92a6a824683e&usertag=55610d65-pygraphistry-0.41.2&splashAfter=1756760532&info=true&pointsOfInterestMax=16'